# LPP (ds003643) - Preprocessing

For each subject and run, 6 simple steps:
1. download the brain file
2. parcellate cortex  -> (T, 1000)   (Schaefer-1000)
3. parcellate subcortex -> (T, 32)    (Tian S2)
4. clean (detrend + z-score)
5. resample to 1 Hz
6. save

Then verify the outputs and delete the raw download to save space.

## Setup

In [1]:
import os, sys

# English (49 subjects; missing: 60,66,71,80,85,90,102,107,111,112)
SUBJECTS = [f"sub-EN{i:03d}" for i in range(57, 116) if i not in (60, 66, 71, 80, 85, 90, 102, 107, 111, 112)]

# Chinese (35 subjects; missing: 12, 35)
# SUBJECTS = [f"sub-CN{i:03d}" for i in range(1, 38) if i not in (12, 35)]

# French (28 subjects; missing: 21, 27)
# SUBJECTS = [f"sub-FR{i:03d}" for i in range(1, 31) if i not in (21, 27)]

PROJECT      = r"C:\TRIBE_Preprocessing"
DATASET      = "lpp"
DATASET_ROOT = os.path.join(PROJECT, "data", DATASET)
ATLAS_DIR    = os.path.join(PROJECT, "atlases")
OUTPUTS_BASE = os.path.join(PROJECT, "outputs", DATASET)

sys.path.append(PROJECT)
import helpers.helpers_lpp as H
print("Subjects:", SUBJECTS)

Subjects: ['sub-EN057', 'sub-EN058', 'sub-EN059', 'sub-EN061', 'sub-EN062', 'sub-EN063', 'sub-EN064', 'sub-EN065', 'sub-EN067', 'sub-EN068', 'sub-EN069', 'sub-EN070', 'sub-EN072', 'sub-EN073', 'sub-EN074', 'sub-EN075', 'sub-EN076', 'sub-EN077', 'sub-EN078', 'sub-EN079', 'sub-EN081', 'sub-EN082', 'sub-EN083', 'sub-EN084', 'sub-EN086', 'sub-EN087', 'sub-EN088', 'sub-EN089', 'sub-EN091', 'sub-EN092', 'sub-EN093', 'sub-EN094', 'sub-EN095', 'sub-EN096', 'sub-EN097', 'sub-EN098', 'sub-EN099', 'sub-EN100', 'sub-EN101', 'sub-EN103', 'sub-EN104', 'sub-EN105', 'sub-EN106', 'sub-EN108', 'sub-EN109', 'sub-EN110', 'sub-EN113', 'sub-EN114', 'sub-EN115']


## Load the two atlases once (cortex = 1000, subcortex = 32)

In [2]:
cortical_atlas    = H.get_cortical_atlas()
subcortical_atlas = H.get_subcortical_atlas(ATLAS_DIR)
H.enable_fallback_remote(DATASET_ROOT)
print("Atlases ready.")

[fetch_atlas_schaefer_2018] Dataset found in C:\TRIBE_Preprocessing\atlases\schaefer_2018
Atlases ready.


## Run the batch

In [3]:
for SUBJECT in SUBJECTS:
    print(f"\n=== {SUBJECT} ===")
    OUT_DIR = os.path.join(OUTPUTS_BASE, SUBJECT)
    try:
        func_dir = H.download_subject(DATASET_ROOT, SUBJECT)   # 1. download
        runs = H.find_runs(func_dir, SUBJECT)
        print(f"  {len(runs)} run(s): {[r['name'] for r in runs]}")

        for run in runs:
            cortex    = H.parcellate(run["bold"], cortical_atlas)      # 2. cortex -> (T,1000)
            subcortex = H.parcellate(run["bold"], subcortical_atlas)   # 3. subcortex -> (T,32)
            cortex    = H.resample_1hz(H.clean(cortex,    run["tr"]), run["tr"])  # 4+5
            subcortex = H.resample_1hz(H.clean(subcortex, run["tr"]), run["tr"])

            tag = f"{SUBJECT}_{run['name']}"
            H.save_npy(OUT_DIR, f"{tag}_cortical_parcels.npy",    cortex)     # 6. save
            H.save_npy(OUT_DIR, f"{tag}_subcortical_parcels.npy", subcortex)
            H.save_metadata(OUT_DIR, f"{tag}_metadata.json", {
                "subject": SUBJECT, "dataset": DATASET, "run": run["name"],
                "native_tr_seconds": run["tr"], "resampled_hz": 1,
                "hemodynamic_lag_seconds": 5,
                "shapes": {"cortical_parcels": list(cortex.shape),
                           "subcortical_parcels": list(subcortex.shape)},
            })
            del cortex, subcortex

        if H.outputs_complete(OUT_DIR, len(runs)):
            print("  outputs verified -> cleaning up")
            H.cleanup_subject(DATASET_ROOT, SUBJECT)
        else:
            print(f"  !! outputs incomplete for {SUBJECT} - keeping raw data")
    except Exception as e:
        print(f"  !! {SUBJECT} FAILED: {e}\n     keeping raw data; moving on.")

print("\n=== BATCH COMPLETE ===")


=== sub-EN057 ===
  downloading 9 file(s) for sub-EN057 ...
  9 run(s): ['task-lppEN_run-15', 'task-lppEN_run-16', 'task-lppEN_run-17', 'task-lppEN_run-18', 'task-lppEN_run-19', 'task-lppEN_run-20', 'task-lppEN_run-21', 'task-lppEN_run-22', 'task-lppEN_run-23']
  saved sub-EN057_task-lppEN_run-15_cortical_parcels.npy  shape=(563, 1000)
  saved sub-EN057_task-lppEN_run-15_subcortical_parcels.npy  shape=(563, 32)
  saved sub-EN057_task-lppEN_run-16_cortical_parcels.npy  shape=(595, 1000)
  saved sub-EN057_task-lppEN_run-16_subcortical_parcels.npy  shape=(595, 32)
  saved sub-EN057_task-lppEN_run-17_cortical_parcels.npy  shape=(679, 1000)
  saved sub-EN057_task-lppEN_run-17_subcortical_parcels.npy  shape=(679, 32)
  saved sub-EN057_task-lppEN_run-18_cortical_parcels.npy  shape=(605, 1000)
  saved sub-EN057_task-lppEN_run-18_subcortical_parcels.npy  shape=(605, 32)
  saved sub-EN057_task-lppEN_run-19_cortical_parcels.npy  shape=(529, 1000)
  saved sub-EN057_task-lppEN_run-19_subcortical_p

## Summary

In [4]:
for SUBJECT in SUBJECTS:
    d = os.path.join(OUTPUTS_BASE, SUBJECT)
    n = len([f for f in os.listdir(d) if f.endswith('.npy')]) if os.path.isdir(d) else 0
    print(f"{SUBJECT}: {n} .npy files")

sub-EN057: 18 .npy files
sub-EN058: 18 .npy files
sub-EN059: 18 .npy files
sub-EN061: 18 .npy files
sub-EN062: 18 .npy files
sub-EN063: 18 .npy files
sub-EN064: 18 .npy files
sub-EN065: 18 .npy files
sub-EN067: 18 .npy files
sub-EN068: 18 .npy files
sub-EN069: 18 .npy files
sub-EN070: 18 .npy files
sub-EN072: 18 .npy files
sub-EN073: 18 .npy files
sub-EN074: 18 .npy files
sub-EN075: 18 .npy files
sub-EN076: 18 .npy files
sub-EN077: 18 .npy files
sub-EN078: 18 .npy files
sub-EN079: 18 .npy files
sub-EN081: 18 .npy files
sub-EN082: 18 .npy files
sub-EN083: 18 .npy files
sub-EN084: 18 .npy files
sub-EN086: 18 .npy files
sub-EN087: 18 .npy files
sub-EN088: 18 .npy files
sub-EN089: 18 .npy files
sub-EN091: 18 .npy files
sub-EN092: 18 .npy files
sub-EN093: 18 .npy files
sub-EN094: 18 .npy files
sub-EN095: 18 .npy files
sub-EN096: 18 .npy files
sub-EN097: 18 .npy files
sub-EN098: 18 .npy files
sub-EN099: 18 .npy files
sub-EN100: 18 .npy files
sub-EN101: 18 .npy files
sub-EN103: 18 .npy files
